In [1]:
import requests
import pandas as pd
import numpy as np
import io
import warnings
warnings.filterwarnings('ignore')

In [2]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import r2_score, mean_absolute_error
from scipy import stats
import joblib

In [3]:
HEADERS = {'User-Agent': 'Mozilla/5.0 (AI-SmartAg-MiniProject/1.0)'}

OWID_URLS = {
    
    'Rice'          : 'https://ourworldindata.org/grapher/rice-yields.csv',
    'Potatoes'      : 'https://ourworldindata.org/grapher/potato-yields.csv',
    'Soybeans'      : 'https://ourworldindata.org/grapher/soybean-yields.csv',
    'Bananas'       : 'https://ourworldindata.org/grapher/banana-yields.csv',
    'Cassava'       : 'https://ourworldindata.org/grapher/cassava-yields.csv',
    'Barley'        : 'https://ourworldindata.org/grapher/barley-yields.csv',
    'Tomatoes'      : 'https://ourworldindata.org/grapher/tomato-yields.csv',
    'Peas'          : 'https://ourworldindata.org/grapher/pea-yields.csv',
    'Beans'         : 'https://ourworldindata.org/grapher/bean-yields.csv',
    'Rapeseed'      : 'https://ourworldindata.org/grapher/rapeseed-yields.csv',
    'Wheat'         : 'https://ourworldindata.org/grapher/wheat-yields.csv',
    'Sorghum'       : 'https://ourworldindata.org/grapher/sorghum-yield.csv',
    'Cotton'        : 'https://ourworldindata.org/grapher/cotton-yield.csv',
    'Maize'         : 'https://ourworldindata.org/grapher/maize-yields.csv',
    'Sugarcane'     : 'https://ourworldindata.org/grapher/sugar-cane-yields.csv',
    'Groundnuts'    : 'https://ourworldindata.org/grapher/groundnuts-yield.csv',
    'Sugarbeet'     : 'https://ourworldindata.org/grapher/sugar-beet-yields.csv',
    'Coffee'        : 'https://ourworldindata.org/grapher/coffee-yields.csv',
    'Oranges'       : 'https://ourworldindata.org/grapher/orange-yields.csv',
    'Palm oil'      : 'https://ourworldindata.org/grapher/palm-oil-yields.csv',
}
print(f'Total crops to fetch: {len(OWID_URLS)}')

for crop, url in OWID_URLS.items():
    try:
        r = requests.get(url, timeout=15)
        print(f'{crop:20s} → {r.status_code}')
    except Exception as e:
        print(f'{crop:20s} → ERROR: {e}')


Total crops to fetch: 20
Rice                 → 200
Potatoes             → 200
Soybeans             → 200
Bananas              → 200
Cassava              → 200
Barley               → 200
Tomatoes             → 200
Peas                 → 200
Beans                → 200
Rapeseed             → 200
Wheat                → 200
Sorghum              → 200
Cotton               → 200
Maize                → 200
Sugarcane            → 200
Groundnuts           → 200
Sugarbeet            → 200
Coffee               → 200
Oranges              → 200
Palm oil             → 200


In [4]:
frames = []
failed = []
for crop, url in OWID_URLS.items():
    try:
        r = requests.get(url, timeout=20)
        if r.status_code == 200:
            tmp = pd.read_csv(io.StringIO(r.text))
            tmp['Item'] = crop
            frames.append(tmp)
            print(f'OK   {crop:20s} {tmp.shape}')
        else:
            failed.append(crop)
            print(f'FAIL {crop} {r.status_code}')
    except Exception as e:
        failed.append(crop)
        print(f'ERR  {crop} {e}')
print(f'Fetched:{len(frames)}  Failed:{len(failed)}')

OK   Rice                 (9870, 5)
OK   Potatoes             (11656, 5)
OK   Soybeans             (7410, 5)
OK   Bananas              (9965, 5)
OK   Cassava              (8270, 5)
OK   Barley               (8110, 5)
OK   Tomatoes             (12138, 5)
OK   Peas                 (7634, 5)
OK   Beans                (9410, 5)
OK   Rapeseed             (5289, 5)
OK   Wheat                (9735, 5)
OK   Sorghum              (8922, 5)
OK   Cotton               (8054, 5)
OK   Maize                (12414, 5)
OK   Sugarcane            (8731, 5)
OK   Groundnuts           (9394, 5)
OK   Sugarbeet            (5146, 5)
OK   Coffee               (7055, 5)
OK   Oranges              (9057, 5)
OK   Palm oil             (4385, 5)
Fetched:20  Failed:0


In [5]:
print(frames[0].columns.tolist())
frames[0].head(3)

['Entity', 'Code', 'Year', 'Rice - Yield (tonnes per hectare)', 'Item']


,Entity,Code,Year,Rice - Yield (tonnes per hectare),Item
0,Afghanistan,AFG,1961,1.519,Rice
1,Afghanistan,AFG,1962,1.519,Rice
2,Afghanistan,AFG,1963,1.519,Rice


In [6]:
clean = []
skip  = {'Entity','Code','Year','Item'}
for raw in frames:
    crop_name = raw['Item'].iloc[0]
    yield_col = [c for c in raw.columns if c not in skip][-1]
    tmp = raw[['Entity','Year', yield_col,'Item']].copy()
    tmp.columns = ['Area','Year','yield_t_ha','Item']
    tmp.dropna(inplace=True)
    tmp = tmp[tmp['yield_t_ha'] > 0]
    tmp['Year'] = tmp['Year'].astype(int)
    tmp['hg/ha_yield'] = tmp['yield_t_ha'] * 10000
    tmp.drop(columns=['yield_t_ha'], inplace=True)
    clean.append(tmp)
owid_df = pd.concat(clean, ignore_index=True)
print(owid_df.shape)
print('Crops:', owid_df['Item'].nunique())
print('Countries:', owid_df['Area'].nunique())

(172629, 4)
Crops: 20
Countries: 251


In [7]:
owid_df = owid_df[(owid_df['Year'] >= 1990) & (owid_df['Year'] <= 2022)]
remove_kw = ['World','Asia','Europe','Africa','America','income',
             'region','OECD','developed','developing','Union','island',
             'Micronesia','Caribbean','Melanesia','Polynesia']
mask = owid_df['Area'].str.contains('|'.join(remove_kw), case=False, na=False)
owid_df = owid_df[~mask]
print(owid_df.shape)
print('Countries:', owid_df['Area'].nunique())

(68764, 4)
Countries: 202


In [8]:
kaggle = pd.read_csv('../data/raw/yield_df.csv', encoding='latin-1')
kaggle.dropna(inplace=True)
print(kaggle.shape)
print(kaggle.columns.tolist())

(28242, 8)
['Unnamed: 0', 'Area', 'Item', 'Year', 'hg/ha_yield', 'average_rain_fall_mm_per_year', 'pesticides_tonnes', 'avg_temp']


In [9]:
climate = kaggle[['Area','Year',
    'average_rain_fall_mm_per_year',
    'pesticides_tonnes',
    'avg_temp']].drop_duplicates(subset=['Area','Year'])
print('Climate lookup:', climate.shape)

Climate lookup: (2250, 5)


In [10]:
df = owid_df.merge(climate, on=['Area','Year'], how='inner')
print('After merge:', df.shape)
print('Crops:', df['Item'].nunique())

After merge: (26552, 7)
Crops: 20


In [11]:
kaggle_clean = kaggle[['Area','Item','Year',
    'average_rain_fall_mm_per_year',
    'pesticides_tonnes','avg_temp','hg/ha_yield']].copy()
owid_crops = df['Item'].unique()
extra = kaggle_clean[~kaggle_clean['Item'].isin(owid_crops)]
print('Extra crops from Kaggle:', extra['Item'].nunique())
print(extra['Item'].unique())

Extra crops from Kaggle: 4
<StringArray>
['Rice, paddy', 'Sweet potatoes', 'Plantains and others', 'Yams']
Length: 4, dtype: str


In [12]:
df_all = pd.concat([df, extra], ignore_index=True)
df_all.dropna(inplace=True)
df_all = df_all[df_all['hg/ha_yield'] > 0]
df_all = df_all[df_all['average_rain_fall_mm_per_year'] > 0]
df_all.drop_duplicates(inplace=True)
print('Combined:', df_all.shape)
print('Crops:', df_all['Item'].nunique())
print('Countries:', df_all['Area'].nunique())

Combined: (33566, 7)
Crops: 24
Countries: 101


In [13]:
df_all['Item'] = df_all['Item'].replace({'Rice, paddy': 'Rice'})
df_all.drop_duplicates(inplace=True)
print('Crops after fix:', df_all['Item'].nunique())

Crops after fix: 23


In [14]:
z = np.abs(stats.zscore(df_all[[
    'hg/ha_yield','average_rain_fall_mm_per_year',
    'avg_temp','pesticides_tonnes'
]]))
df_all = df_all[(z < 3).all(axis=1)]
print('After outlier removal:', df_all.shape)

After outlier removal: (32208, 7)


In [15]:
crops = sorted(df_all['Item'].unique())
print(f'Total crops ({len(crops)}):')
for c in crops:
    print(f'  "{c}"')

Total crops (23):
  "Bananas"
  "Barley"
  "Beans"
  "Cassava"
  "Coffee"
  "Cotton"
  "Groundnuts"
  "Maize"
  "Oranges"
  "Palm oil"
  "Peas"
  "Plantains and others"
  "Potatoes"
  "Rapeseed"
  "Rice"
  "Sorghum"
  "Soybeans"
  "Sugarbeet"
  "Sugarcane"
  "Sweet potatoes"
  "Tomatoes"
  "Wheat"
  "Yams"


In [16]:
df_all.to_csv('../data/processed/yield_combined_dataset.csv', index=False)
print('Saved')

Saved


In [17]:
X = df_all[['Area','Item','Year',
    'average_rain_fall_mm_per_year','pesticides_tonnes','avg_temp']]
y = df_all['hg/ha_yield']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)
print(X_train.shape, X_test.shape)

(25766, 6) (6442, 6)


In [18]:
preprocessor = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), ['Area','Item'])
], remainder='passthrough')

In [19]:
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(
        n_estimators=200, max_depth=20,
        min_samples_leaf=3, n_jobs=1, random_state=42
    ))
])
pipeline.fit(X_train, y_train)
print('Done')

Done


In [20]:
y_pred = pipeline.predict(X_test)
print('Test R²  :', round(r2_score(y_test, y_pred), 4))
print('Train R² :', round(r2_score(y_train, pipeline.predict(X_train)), 4))
print('Test MAE :', round(mean_absolute_error(y_test, y_pred)/10, 2), 'kg/ha')

Test R²  : 0.9607
Train R² : 0.9845
Test MAE : 1329.48 kg/ha


In [21]:
X_s = X.sample(n=5000, random_state=42)
y_s = y[X_s.index]
cv  = cross_val_score(pipeline, X_s, y_s, cv=3, scoring='r2', n_jobs=1)
print('CV R²:', round(cv.mean(),4), '±', round(cv.std(),4))

CV R²: 0.8444 ± 0.0075


In [22]:
sample = X_test.iloc[:5].copy()
sample['actual_kg']    = y_test.iloc[:5].values / 10
sample['predicted_kg'] = pipeline.predict(X_test.iloc[:5]) / 10
sample[['Area','Item','Year','actual_kg','predicted_kg']]

,Area,Item,Year,actual_kg,predicted_kg
11575,Turkey,Peas,2000,2330.8000,2204.966088
3402,Ukraine,Potatoes,2010,13248.1000,21978.654480
30135,India,Sweet potatoes,2002,8569.4000,8768.100355
6960,Brazil,Barley,1993,1639.5000,1359.139864
13863,Lithuania,Rapeseed,2010,1654.2001,1818.749580


In [23]:
joblib.dump(pipeline, '../backend/models/crop_yield_model.pkl')
print('Model saved → backend/models/crop_yield_model.pkl')
print(f'Crops      : {df_all["Item"].nunique()}')
print(f'Countries  : {df_all["Area"].nunique()}')
print(f'Total rows : {len(df_all)}')

Model saved → backend/models/crop_yield_model.pkl
Crops      : 23
Countries  : 101
Total rows : 32208


In [24]:
crops = sorted(df_all['Item'].unique())
print(f'Total ({len(crops)}):')
for c in crops:
    print(f'  "{c.lower()}" : "{c}",')

Total (23):
  "bananas" : "Bananas",
  "barley" : "Barley",
  "beans" : "Beans",
  "cassava" : "Cassava",
  "coffee" : "Coffee",
  "cotton" : "Cotton",
  "groundnuts" : "Groundnuts",
  "maize" : "Maize",
  "oranges" : "Oranges",
  "palm oil" : "Palm oil",
  "peas" : "Peas",
  "plantains and others" : "Plantains and others",
  "potatoes" : "Potatoes",
  "rapeseed" : "Rapeseed",
  "rice" : "Rice",
  "sorghum" : "Sorghum",
  "soybeans" : "Soybeans",
  "sugarbeet" : "Sugarbeet",
  "sugarcane" : "Sugarcane",
  "sweet potatoes" : "Sweet potatoes",
  "tomatoes" : "Tomatoes",
  "wheat" : "Wheat",
  "yams" : "Yams",
